# EDA — Dataset sintético embarazo (solo tablas)

Exploración tabular del dataset generado. Sin gráficos.

**Fuentes:**
- `../data/datos_embarazo_sintetico.csv`
- `../data/metadatos_ground_truth.csv`
- `../data/missingness_log.csv`

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")

df = pd.read_csv(DATA_DIR / "v2/clustering_feature_view.csv")
meta = pd.read_csv(DATA_DIR / "v2/ground_truth.csv")
log_miss = pd.read_csv(DATA_DIR / "v2/missingness_log.csv")

df_full = df.merge(meta, on="patient_id", how="left")

NUMERIC = df.select_dtypes(include=[np.number]).columns.tolist()
CATEGORIC = df.select_dtypes(include=["object"]).columns.tolist()
BINARY = [c for c in NUMERIC if set(df[c].dropna().unique()).issubset({0, 0.0, 1, 1.0})]
CONTINUOUS = [c for c in NUMERIC if c not in BINARY and c != "patient_id"]

C:\Users\MAXIMILIANO\AppData\Local\Temp\ipykernel_13432\1319048061.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  CATEGORIC = df.select_dtypes(include=["object"]).columns.tolist()


## 1. Dimensiones y tipos

In [11]:
resumen_general = pd.DataFrame({
    "metrica": ["Filas", "Columnas features", "Variables numéricas", "Variables categóricas", "Duplicados patient_id"],
    "valor": [len(df), len(df.columns) - 1, len(NUMERIC), len(CATEGORIC), df["patient_id"].duplicated().sum()],
})
display(resumen_general)

display(pd.DataFrame({
    "columna": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "no_nulos": df.notna().sum().values,
    "pct_completo": (100 * df.notna().mean()).round(2).values,
}))

,metrica,valor
0,Filas,2000
1,Columnas features,27
2,Variables numéricas,25
3,Variables categóricas,3
4,Duplicados patient_id,0


,columna,dtype,no_nulos,pct_completo
0,patient_id,int64,2000,100.00
1,age_years,int64,2000,100.00
2,bmi_initial,float64,2000,100.00
3,gestational_week,float64,2000,100.00
4,gestational_trimester,int64,2000,100.00
5,height_cm,float64,1784,89.20
6,initial_weight,float64,1930,96.50
7,weight_kg,float64,1862,93.10
8,weight_gain,float64,2000,100.00
9,systolic,float64,1859,92.95


## 2. Muestra de registros

In [12]:
display(df_full.head(10))
display(df_full.tail(5))

,patient_id,age_years,bmi_initial,gestational_week,gestational_trimester,height_cm,initial_weight,weight_kg,weight_gain,systolic,diastolic,mean_arterial_pressure,diabetes,chronic_hypertension,previous_preeclampsia,family_history_hypertension,family_history_heart_disease,chronic_kidney_disease,multiple_pregnancy,active_smoking,previous_pregnancies,previous_deliveries,previous_miscarriages,previous_cesareans,nulliparous,education_level,residence,marital_status,cluster_verdadero,es_outlier,tipo_outlier
0,1,19,18.41,32.80,3,152.80,43.00,48.10,5.10,113.00,58.00,76.70,0,0,0,0,0,0,0,0,1.00,0,1,0,0,primaria,rural,single,3,0,NaN
1,2,31,19.21,34.20,3,161.30,50.00,59.20,9.20,115.00,68.00,83.80,0,0,0,0,0,0,0,0,1.00,0,1,0,0,superior,urbana,married,0,0,NaN
2,3,26,17.80,28.90,3,159.50,45.30,52.00,6.70,108.00,65.00,79.40,0,0,0,0,1,0,0,0,0.00,0,0,0,1,primaria,urbana,married,3,0,NaN
3,4,42,26.48,24.50,2,159.80,67.60,72.60,5.00,145.00,96.00,112.10,0,1,0,1,0,0,0,0,1.00,0,1,0,0,superior,urbana,single,2,0,NaN
4,5,32,21.01,40.00,3,161.80,55.00,NaN,11.70,113.00,66.00,81.70,0,0,0,0,0,0,0,0,2.00,0,2,0,0,superior,urbana,single,0,0,NaN
5,6,27,28.72,28.40,3,151.30,65.80,71.20,5.40,103.00,73.00,83.00,0,0,0,0,0,0,0,0,0.00,0,0,0,1,superior,rural,married,5,0,NaN
6,7,26,17.89,37.50,3,153.10,41.90,46.80,4.90,96.00,73.00,80.50,0,0,0,0,0,0,0,0,4.00,0,4,0,0,primaria,urbana,single,3,0,NaN
7,8,25,15.79,33.00,3,155.70,38.30,43.40,5.10,106.00,66.00,79.20,0,0,0,0,0,0,0,0,0.00,0,0,0,1,superior,urbana,single,3,0,NaN
8,9,31,24.11,30.60,3,160.30,61.90,69.70,7.80,114.00,69.00,84.30,0,0,0,0,0,0,0,0,4.00,2,2,2,0,superior,urbana,married,0,0,NaN
9,10,36,31.72,32.00,3,158.00,79.20,NaN,10.00,129.00,76.00,93.90,1,0,0,0,1,0,0,0,2.00,0,2,0,0,superior,urbana,single,1,0,NaN


,patient_id,age_years,bmi_initial,gestational_week,gestational_trimester,height_cm,initial_weight,weight_kg,weight_gain,systolic,diastolic,mean_arterial_pressure,diabetes,chronic_hypertension,previous_preeclampsia,family_history_hypertension,family_history_heart_disease,chronic_kidney_disease,multiple_pregnancy,active_smoking,previous_pregnancies,previous_deliveries,previous_miscarriages,previous_cesareans,nulliparous,education_level,residence,marital_status,cluster_verdadero,es_outlier,tipo_outlier
1995,1996,29,18.11,40.00,3,168.00,51.10,55.10,4.00,110.00,67.00,81.40,0,0,0,0,0,0,0,1,0.00,0,0,0,1,secundaria,urbana,married,3,0,NaN
1996,1997,33,24.93,28.70,3,156.00,60.70,65.80,5.10,105.00,66.00,78.80,0,0,0,0,0,0,0,0,3.00,0,3,0,0,superior,rural,married,0,0,NaN
1997,1998,25,25.46,32.90,3,145.30,53.70,61.50,7.70,112.00,78.00,89.20,0,0,0,1,0,0,0,0,5.00,0,5,0,0,secundaria,urbana,single,0,0,NaN
1998,1999,27,18.84,38.80,3,163.10,50.10,56.90,6.80,105.00,65.00,78.60,0,0,0,0,0,0,0,0,0.00,0,0,0,1,primaria,urbana,married,3,0,NaN
1999,2000,26,29.12,29.80,3,150.10,65.60,75.30,9.70,110.00,81.00,90.80,0,0,0,0,0,0,0,0,4.00,4,0,3,0,superior,urbana,married,1,0,NaN


## 3. Estadísticos descriptivos — variables continuas

In [13]:
desc = df[CONTINUOUS].describe().T
desc["IQR"] = desc["75%"] - desc["25%"]
desc["CV"] = (desc["std"] / desc["mean"]).replace([np.inf, -np.inf], np.nan)
display(desc.round(2))

,count,mean,std,min,25%,50%,75%,max,IQR,CV
age_years,"2,000.00",29.12,5.91,15.00,25.00,29.00,33.00,49.00,8.00,0.20
bmi_initial,"2,000.00",24.88,5.24,14.67,21.45,24.11,28.26,40.36,6.81,0.21
gestational_week,"2,000.00",31.41,4.96,14.00,28.50,31.60,34.90,40.00,6.40,0.16
gestational_trimester,"2,000.00",2.83,0.38,2.00,3.00,3.00,3.00,3.00,0.00,0.13
height_cm,"1,784.00",174.81,149.76,140.00,156.17,160.20,164.52,"1,700.00",8.35,0.86
initial_weight,"1,930.00",63.67,14.12,33.60,53.72,61.80,72.10,112.70,18.37,0.22
weight_kg,"1,862.00",71.31,15.52,38.70,61.10,69.45,80.40,128.70,19.30,0.22
weight_gain,"2,000.00",7.65,3.45,-8.00,5.50,7.50,9.80,19.10,4.30,0.45
systolic,"1,859.00",120.97,20.48,92.00,108.00,114.00,129.00,229.52,21.00,0.17
diastolic,"1,941.00",75.66,12.19,52.00,67.00,72.00,84.00,115.00,17.00,0.16


## 4. Percentiles detallados

In [14]:
percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
pct_table = df[CONTINUOUS].quantile(percentiles).T
pct_table.columns = [f"p{int(p*100)}" for p in percentiles]
display(pct_table.round(2))

,p1,p5,p10,p25,p50,p75,p90,p95,p99
age_years,16.00,19.00,22.00,25.00,29.00,33.00,37.00,40.00,43.01
bmi_initial,15.91,17.07,18.00,21.45,24.11,28.26,32.55,34.56,38.00
gestational_week,17.60,22.80,24.90,28.50,31.60,34.90,37.80,39.40,40.00
gestational_trimester,2.00,2.00,2.00,3.00,3.00,3.00,3.00,3.00,3.00
height_cm,145.90,150.00,152.10,156.17,160.20,164.52,168.10,170.80,179.60
initial_weight,38.63,42.80,45.60,53.72,61.80,72.10,84.11,89.36,99.14
weight_kg,43.36,47.50,51.70,61.10,69.45,80.40,93.59,99.90,110.29
weight_gain,-0.50,2.60,3.79,5.50,7.50,9.80,11.80,13.50,15.90
systolic,97.00,101.00,103.80,108.00,114.00,129.00,147.00,155.00,215.38
diastolic,57.00,61.00,63.00,67.00,72.00,84.00,95.00,99.00,105.00


## 5. Variables binarias — frecuencias

In [15]:
filas_bin = []
for col in BINARY:
    vc = df[col].value_counts(dropna=False)
    for val, cnt in vc.items():
        filas_bin.append({
            "variable": col,
            "valor": val,
            "conteo": cnt,
            "pct": round(100 * cnt / len(df), 2),
        })
display(pd.DataFrame(filas_bin).sort_values(["variable", "valor"]))

,variable,valor,conteo,pct
14,active_smoking,0,1820,91.00
15,active_smoking,1,180,9.00
2,chronic_hypertension,0,1786,89.30
3,chronic_hypertension,1,214,10.70
10,chronic_kidney_disease,0,1948,97.40
11,chronic_kidney_disease,1,52,2.60
0,diabetes,0,1771,88.55
1,diabetes,1,229,11.45
8,family_history_heart_disease,0,1789,89.45
9,family_history_heart_disease,1,211,10.55


## 6. Variables categóricas — frecuencias

In [16]:
filas_cat = []
for col in CATEGORIC:
    vc = df[col].value_counts(dropna=False)
    for val, cnt in vc.items():
        filas_cat.append({
            "variable": col,
            "categoria": val,
            "conteo": cnt,
            "pct": round(100 * cnt / len(df), 2),
        })
display(pd.DataFrame(filas_cat).sort_values(["variable", "conteo"], ascending=[True, False]))

,variable,categoria,conteo,pct
0,education_level,superior,893,44.65
1,education_level,secundaria,733,36.65
2,education_level,primaria,374,18.70
5,marital_status,married,1195,59.75
6,marital_status,single,805,40.25
3,residence,urbana,1515,75.75
4,residence,rural,485,24.25


## 7. Valores faltantes por columna

In [17]:
missing = pd.DataFrame({
    "variable": df.columns,
    "nulos": df.isna().sum().values,
    "pct_nulos": (100 * df.isna().mean()).round(2).values,
    "completos": df.notna().sum().values,
}).sort_values("nulos", ascending=False)
display(missing)

filas_con_algun_nulo = df.isna().any(axis=1).sum()
display(pd.DataFrame({
    "metrica": ["Filas con al menos un nulo", "Filas completas"],
    "conteo": [filas_con_algun_nulo, len(df) - filas_con_algun_nulo],
    "pct": [round(100 * filas_con_algun_nulo / len(df), 2), round(100 * (len(df) - filas_con_algun_nulo) / len(df), 2)],
}))

,variable,nulos,pct_nulos,completos
5,height_cm,216,10.80,1784
9,systolic,141,7.05,1859
7,weight_kg,138,6.90,1862
6,initial_weight,70,3.50,1930
10,diastolic,59,2.95,1941
20,previous_pregnancies,54,2.70,1946
1,age_years,0,0.00,2000
2,bmi_initial,0,0.00,2000
3,gestational_week,0,0.00,2000
4,gestational_trimester,0,0.00,2000


,metrica,conteo,pct
0,Filas con al menos un nulo,588,29.40
1,Filas completas,1412,70.60


## 8. Missingness por mecanismo (ground truth)

In [18]:
display(log_miss.groupby("mecanismo").size().reset_index(name="celdas_afectadas"))
display(
    log_miss.groupby(["variable", "mecanismo"])
    .size()
    .reset_index(name="conteo")
    .sort_values("conteo", ascending=False)
)

,mecanismo,celdas_afectadas
0,mar,247
1,mcar,360
2,mnar,81


,variable,mecanismo,conteo
1,height_cm,mar,160
7,weight_kg,mar,87
6,systolic,mnar,81
3,initial_weight,mcar,70
2,height_cm,mcar,65
5,systolic,mcar,60
0,diastolic,mcar,59
4,previous_pregnancies,mcar,54
8,weight_kg,mcar,52


## 9. Outliers inyectados (ground truth)

In [19]:
display(meta.groupby("es_outlier").size().reset_index(name="pacientes"))
display(
    meta[meta["es_outlier"] == 1]
    .groupby("tipo_outlier")
    .size()
    .reset_index(name="conteo")
    .sort_values("conteo", ascending=False)
)
display(
    meta.groupby(["cluster_verdadero", "es_outlier"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "sin_outlier", 1: "con_outlier"})
)

,es_outlier,pacientes
0,0,1900
1,1,100


,tipo_outlier,conteo
1,medicion,31
4,registro,20
2,patologia_rara,19
3,posicion,17
0,combinacion_imposible,13


es_outlier,sin_outlier,con_outlier
cluster_verdadero,,
0,849,48
1,337,14
2,216,12
3,267,11
4,132,9
5,99,6


## 10. Distribución de clusters verdaderos

In [20]:
CLUSTER_LABELS = {
    0: "C0 — Bajo riesgo",
    1: "C1 — Riesgo metabólico",
    2: "C2 — Riesgo hipertensivo",
    3: "C3 — Bajo peso nutricional",
    4: "C4 — Alto riesgo obstétrico",
    5: "C5 — Residual",
}

dist_cluster = meta["cluster_verdadero"].value_counts().sort_index().reset_index()
dist_cluster.columns = ["cluster", "conteo"]
dist_cluster["pct"] = (100 * dist_cluster["conteo"] / len(meta)).round(2)
dist_cluster["perfil"] = dist_cluster["cluster"].map(CLUSTER_LABELS)
display(dist_cluster)

,cluster,conteo,pct,perfil
0,0,897,44.85,C0 — Bajo riesgo
1,1,351,17.55,C1 — Riesgo metabólico
2,2,228,11.40,C2 — Riesgo hipertensivo
3,3,278,13.90,C3 — Bajo peso nutricional
4,4,141,7.05,C4 — Alto riesgo obstétrico
5,5,105,5.25,C5 — Residual


## 11. Estadísticos por cluster verdadero

In [21]:
vars_perfil = [
    "age_years", "bmi_initial", "gestational_week",
    "weight_gain", "systolic", "diastolic", "mean_arterial_pressure",
]

por_cluster = (
    df_full.groupby("cluster_verdadero")[vars_perfil]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .round(2)
)
display(por_cluster)

age_years                           bmi_initial             \
                      count  mean  std min median max       count  mean  std   
cluster_verdadero                                                              
0                       897 27.79 4.24  16  28.00  42         897 23.08 2.33   
1                       351 29.87 5.18  16  30.00  42         351 31.98 3.06   
2                       228 32.06 5.67  16  32.00  47         228 25.97 3.20   
3                       278 24.05 4.97  15  24.00  38         278 17.53 1.62   
4                       141 37.83 4.86  17  38.00  49         141 28.85 4.43   
5                       105 33.35 5.71  17  34.00  45         105 28.37 3.83   

                                     gestational_week                          \
                    min median   max            count  mean  std   min median   
cluster_verdadero                                                               
0                 15.60  22.93 38.00              897 31.99 4.09 19.90  31.90   
1                 24.13  31.94 40.36              351 30.58 5.48 14.00  30.30   
2                 16.97  26.14 38.00              228 28.07 6.48 14.00  28.45   
3                 14.67  17.40 38.00              278 33.45 3.56 23.00  33.40   
4                 19.26  28.66 38.18              141 31.72 4.68 19.70  31.70   
5                 20.49  28.21 38.30              105 30.76 5.65 16.10  30.80   

                        weight_gain                               systolic  \
                    max       count  mean  std   min median   max    count   
cluster_verdadero                                                            
0                 40.00         897  7.91 2.58 -8.00   8.00 15.10      866   
1                 40.00         351 10.39 3.66 -8.00  10.60 19.10      329   
2                 40.00         228  5.06 2.83 -8.00   5.00 11.80      193   
3                 40.00         278  5.01 1.84 -8.00   5.00 10.00      261   
4                 40.00         141  8.17 3.77 -8.00   8.50 16.40      126   
5                 40.00         105  8.08 4.58 -1.70   8.00 16.90       84   

                                                    diastolic              \
                    mean   std    min median    max     count  mean   std   
cluster_verdadero                                                           
0                 112.69 16.03  94.00 110.00 229.52       872 68.33  5.15   
1                 126.22 12.76  99.00 125.00 228.13       339 80.35  6.09   
2                 149.33 14.16 100.00 148.00 226.34       223 95.32  6.58   
3                 107.96 15.76  92.00 106.00 228.29       268 65.40  5.95   
4                 136.49 16.69 100.00 135.50 221.43       137 89.36  7.71   
5                 137.82 22.88 100.00 136.00 224.46       102 88.26 12.25   

                                      mean_arterial_pressure               \
                    min median    max                  count   mean   std   
cluster_verdadero                                                           
0                 54.00  68.00 100.00                    897  82.44  4.66   
1                 60.00  80.00 100.00                    351  95.55  4.88   
2                 60.00  96.00 112.00                    228 113.27  4.57   
3                 52.00  65.00  99.00                    278  78.87  4.99   
4                 60.00  90.00 107.00                    141 104.94  6.69   
5                 54.00  89.50 115.00                    105 104.63 11.39   

                                        
                     min median    max  
cluster_verdadero                       
0                  70.90  82.00 116.40  
1                  86.00  95.20 114.10  
2                 101.80 113.30 127.60  
3                  66.50  78.70 113.00  
4                  86.10 105.70 119.60  
5                  80.10 106.30 128.50

## 12. Tablas cruzadas

In [22]:
display(pd.crosstab(df_full["cluster_verdadero"], df_full["gestational_trimester"], margins=True))
display(pd.crosstab(df_full["cluster_verdadero"], df_full["diabetes"], margins=True))
display(pd.crosstab(df_full["cluster_verdadero"], df_full["chronic_hypertension"], margins=True))
display(pd.crosstab(df_full["education_level"], df_full["residence"], margins=True))

gestational_trimester,2,3,All
cluster_verdadero,,,
0,97,800,897
1,89,262,351
2,96,132,228
3,9,269,278
4,24,117,141
5,27,78,105
All,342,1658,2000


diabetes,0,1,All
cluster_verdadero,,,
0,884,13,897
1,211,140,351
2,214,14,228
3,274,4,278
4,112,29,141
5,76,29,105
All,1771,229,2000


chronic_hypertension,0,1,All
cluster_verdadero,,,
0,883,14,897
1,334,17,351
2,109,119,228
3,275,3,278
4,107,34,141
5,78,27,105
All,1786,214,2000


residence,rural,urbana,All
education_level,,,
primaria,141,233,374
secundaria,183,550,733
superior,161,732,893
All,485,1515,2000


## 13. Matriz de correlación (Pearson)

In [23]:
corr = df[CONTINUOUS].corr(method="pearson").round(3)
display(corr)

pares = []
cols = corr.columns.tolist()
for i, a in enumerate(cols):
    for b in cols[i + 1:]:
        pares.append({"var_a": a, "var_b": b, "r": corr.loc[a, b]})
pares_df = pd.DataFrame(pares).sort_values("r", key=abs, ascending=False)
display(pares_df.head(15))

,age_years,bmi_initial,gestational_week,gestational_trimester,height_cm,initial_weight,weight_kg,weight_gain,systolic,diastolic,mean_arterial_pressure,previous_pregnancies,previous_deliveries,previous_miscarriages,previous_cesareans
age_years,1.00,0.35,-0.07,-0.11,0.01,0.33,0.32,0.08,0.27,0.43,0.45,0.09,0.04,0.08,0.04
bmi_initial,0.35,1.00,-0.12,-0.16,0.02,0.92,0.91,0.28,0.34,0.47,0.49,0.06,0.04,0.04,0.02
gestational_week,-0.07,-0.12,1.00,0.72,0.03,-0.13,-0.00,0.50,-0.19,-0.22,-0.25,-0.02,-0.02,0.01,-0.01
gestational_trimester,-0.11,-0.16,0.72,1.00,-0.00,-0.16,-0.07,0.37,-0.22,-0.24,-0.27,-0.01,-0.02,0.01,-0.02
height_cm,0.01,0.02,0.03,-0.00,1.00,0.04,0.05,0.05,-0.04,0.03,0.02,0.01,-0.03,0.06,-0.02
initial_weight,0.33,0.92,-0.13,-0.16,0.04,1.00,0.98,0.34,0.32,0.46,0.48,0.07,0.03,0.06,0.02
weight_kg,0.32,0.91,-0.00,-0.07,0.05,0.98,1.00,0.50,0.30,0.41,0.43,0.07,0.04,0.05,0.02
weight_gain,0.08,0.28,0.50,0.37,0.05,0.34,0.50,1.00,-0.01,-0.01,-0.02,0.05,0.02,0.04,0.03
systolic,0.27,0.34,-0.19,-0.22,-0.04,0.32,0.30,-0.01,1.00,0.60,0.69,0.03,-0.00,0.04,-0.00
diastolic,0.43,0.47,-0.22,-0.24,0.03,0.46,0.41,-0.01,0.60,1.00,0.96,0.04,0.02,0.03,0.04


,var_a,var_b,r
60,initial_weight,weight_kg,0.98
90,diastolic,mean_arterial_pressure,0.96
17,bmi_initial,initial_weight,0.92
18,bmi_initial,weight_kg,0.91
103,previous_deliveries,previous_cesareans,0.74
27,gestational_week,gestational_trimester,0.72
100,previous_pregnancies,previous_miscarriages,0.70
85,systolic,mean_arterial_pressure,0.69
99,previous_pregnancies,previous_deliveries,0.68
84,systolic,diastolic,0.60


## 14. Detección de atípicos univariados (regla IQR)

In [24]:
iqr_rows = []
for col in CONTINUOUS:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    iqr_rows.append({
        "variable": col,
        "Q1": round(q1, 2),
        "Q3": round(q3, 2),
        "IQR": round(iqr, 2),
        "limite_inf": round(lo, 2),
        "limite_sup": round(hi, 2),
        "n_atipicos_iqr": int(n_out),
        "pct_atipicos": round(100 * n_out / len(df), 2),
    })
display(pd.DataFrame(iqr_rows).sort_values("n_atipicos_iqr", ascending=False))

,variable,Q1,Q3,IQR,limite_inf,limite_sup,n_atipicos_iqr,pct_atipicos
3,gestational_trimester,3.00,3.00,0.00,3.00,3.00,342,17.10
14,previous_cesareans,0.00,1.00,1.00,-1.50,2.50,102,5.10
8,systolic,108.00,129.00,21.00,76.50,160.50,59,2.95
7,weight_gain,5.50,9.80,4.30,-0.95,16.25,36,1.80
2,gestational_week,28.50,34.90,6.40,18.90,44.50,27,1.35
4,height_cm,156.17,164.52,8.35,143.65,177.05,24,1.20
6,weight_kg,61.10,80.40,19.30,32.15,109.35,23,1.15
5,initial_weight,53.72,72.10,18.37,26.16,99.66,19,0.95
1,bmi_initial,21.45,28.26,6.81,11.23,38.49,9,0.45
0,age_years,25.00,33.00,8.00,13.00,45.00,6,0.30


## 15. Perfil por gestational_trimester gestacional

In [25]:
por_gestational_trimester = (
    df.groupby("gestational_trimester")[vars_perfil]
    .agg(["count", "mean", "std"])
    .round(2)
)
display(por_gestational_trimester)

age_years            bmi_initial             \
                          count  mean  std       count  mean  std   
gestational_trimester                                               
2                           342 30.57 5.95         342 26.75 4.83   
3                          1658 28.82 5.86        1658 24.50 5.24   

                      gestational_week            weight_gain            \
                                 count  mean  std       count mean  std   
gestational_trimester                                                     
2                                  342 23.58 3.12         342 4.83 2.73   
3                                 1658 33.03 3.52        1658 8.23 3.30   

                      systolic              diastolic              \
                         count   mean   std     count  mean   std   
gestational_trimester                                               
2                          312 130.92 23.59       330 82.25 12.82   
3                         1547 118.97 19.18      1611 74.31 11.61   

                      mean_arterial_pressure              
                                       count  mean   std  
gestational_trimester                                     
2                                        342 98.05 13.61  
3                                       1658 88.96 12.18

## 16. Coherencia clínica básica (reglas duras)

In [26]:
validaciones = []

pa_invalida = df.dropna(subset=["systolic", "diastolic"])
pa_invalida = pa_invalida[pa_invalida["systolic"] <= pa_invalida["diastolic"] + 14]
validaciones.append({"regla": "PA sistólica > diastólica + 14", "violaciones": len(pa_invalida)})

paridad = df.dropna(subset=["previous_pregnancies", "previous_deliveries"])
paridad_bad = paridad[paridad["previous_deliveries"] > paridad["previous_pregnancies"]]
validaciones.append({"regla": "partos <= embarazos previos", "violaciones": len(paridad_bad)})

trim_bad = df[df["gestational_trimester"] != df["gestational_week"].apply(
    lambda s: 1 if s <= 13 else (2 if s <= 27 else 3) if pd.notna(s) else np.nan
)]
validaciones.append({"regla": "gestational_trimester coherente con semanas", "violaciones": len(trim_bad)})

display(pd.DataFrame(validaciones))

,regla,violaciones
0,PA sistólica > diastólica + 14,0
1,partos <= embarazos previos,0
2,gestational_trimester coherente con semanas,3
